# 🛡️ DeepGuard AI — GPU Training on Colab

**One-click training pipeline** for the DeepGuard AI deepfake detection model.

- **Runtime**: Go to `Runtime > Change runtime type > T4 GPU`
- **Time**: ~15-30 minutes on T4 GPU (vs 3-4 hours on CPU)
- **Output**: Trained `.keras` model downloaded to your PC

---

## Step 0: Verify GPU

In [ ]:
import tensorflow as tf
print(f'TensorFlow: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs available: {len(gpus)}')
for gpu in gpus:
    print(f'  {gpu}')
if not gpus:
    print('\n[WARNING] No GPU detected!')
    print('Go to Runtime > Change runtime type > T4 GPU')

## Step 1: Clone the repo

In [ ]:
!git clone https://github.com/vishnuwadkar/Deepfake-Detection-Project.git
%cd Deepfake-Detection-Project

## Step 2: Install dependencies

In [ ]:
!pip install -q mtcnn tqdm scikit-learn

## Step 3: Setup Kaggle API & Download Dataset

Upload your `kaggle.json` file when prompted.

In [ ]:
# Upload kaggle.json
from google.colab import files
uploaded = files.upload()  # Upload your kaggle.json here

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print('Kaggle API configured!')

In [ ]:
# Download the 140K Real-and-Fake-Faces dataset
!pip install -q kaggle
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces -p data/raw --unzip

# Organize into processed directory
import shutil
from pathlib import Path

src_root = Path('data/raw/real_vs_fake/real-vs-fake')
dst_root = Path('data/processed')

for split in ['train', 'valid', 'test']:
    for cls in ['real', 'fake']:
        src = src_root / split / cls
        dst = dst_root / split / cls
        if not dst.exists() and src.exists():
            shutil.copytree(str(src), str(dst))
            print(f'  {split}/{cls}: {len(list(dst.glob("*"))):,} files')

print('\nDataset ready!')

## Step 4: Configure for Full GPU Training

Override config to use **all 100K training images** (GPU can handle it).

In [ ]:
# Override config for GPU: use ALL data, more epochs
import src.config as cfg

cfg.MAX_TRAIN_SAMPLES = None   # Use all 100K images
cfg.BATCH_SIZE = 64            # Optimal for T4 GPU
cfg.EPOCHS_HEAD = 12           # More epochs with full data
cfg.EPOCHS_FINETUNE = 15       # Fine-tuning phase

print(f'Training config:')
print(f'  MAX_TRAIN_SAMPLES: {cfg.MAX_TRAIN_SAMPLES} (all data)')
print(f'  BATCH_SIZE: {cfg.BATCH_SIZE}')
print(f'  Phase 1: {cfg.EPOCHS_HEAD} epochs')
print(f'  Phase 2: {cfg.EPOCHS_FINETUNE} epochs')

## Step 5: Train the Model

This will run the full two-phase training pipeline (~15-30 min on T4).

In [ ]:
from src.train import train
train()

## Step 6: Evaluate

In [ ]:
from src.evaluate import evaluate
evaluate()

# Show the evaluation report
from IPython.display import Image, display
display(Image('models/evaluation_report.png', width=900))

## Step 7: Download the Trained Model

Download the model to your PC, then place it in your project's `models/` folder.

In [ ]:
from google.colab import files
import os

model_path = 'models/deepfake_detector.keras'
if os.path.exists(model_path):
    size_mb = os.path.getsize(model_path) / (1024*1024)
    print(f'Model size: {size_mb:.1f} MB')
    files.download(model_path)
    print('\nDownload started! Place this file in your project\'s models/ folder.')
else:
    print('Model not found. Training may have failed.')

# Also download the training history plot
history_path = 'models/training_history.png'
if os.path.exists(history_path):
    files.download(history_path)

# And evaluation report
eval_path = 'models/evaluation_report.png'
if os.path.exists(eval_path):
    files.download(eval_path)

## Step 8 (Optional): Convert to TF.js for Chrome Extension

In [ ]:
!pip install -q tensorflowjs
!python scripts/convert_model.py

# Download the TF.js model files
import zipfile
with zipfile.ZipFile('tfjs_model.zip', 'w') as z:
    model_dir = Path('extension/model')
    for f in model_dir.iterdir():
        z.write(f, f.name)

files.download('tfjs_model.zip')
print('\nExtract this zip into your extension/model/ folder.')